# Laboratorio 2 — Regresión Logística Binaria
## Dataset: CICIoT2023 — Detección de Ataques en Redes IoT

Aplicado al conjunto de datos **CICIoT2023**, que contiene capturas de tráfico de red de dispositivos IoT etiquetadas con 34 categorías de ataque y tráfico benigno.

- **Tarea:** Clasificación binaria — detectar si un flujo de red es un **ataque (1)** o **tráfico benigno (0)**
- **Variable objetivo (y):** `0 = BenignTraffic`, `1 = Ataque` (DDoS, DoS, Mirai, Recon, etc.)
- **Variables de entrada (X):** 46 características de tráfico de red (tamaños de paquetes, flags TCP, protocolos, estadísticas de flujo) — **n = 46 ≥ 5**
- **Ejemplos (m):** 60,000 muestras balanceadas (30,000 benignas + 30,000 ataques) — **m = 60,000 ≥ 20,000**
- **Preprocesamiento:** realizado con **pandas**
- **División:** 80% entrenamiento / 20% validación

In [ ]:
# utilizado para el manejo de rutas y directorios
import os

# Cálculo científico y vectorial para python
import numpy as np

# Librería para graficación
from matplotlib import pyplot

# Librería para manipulación y preprocesamiento de datos
import pandas as pd

## 1. Carga y Preprocesamiento del Dataset

Se utiliza **pandas** para cargar el archivo CSV y realizar el preprocesamiento:

1. Eliminar la columna de índice innecesaria.
2. Eliminar filas con valores nulos.
3. Convertir la columna `label` a binaria: `BenignTraffic → 0`, cualquier tipo de ataque → `1`.
4. **Balancear clases:** se usan 30,000 ejemplos de tráfico benigno y 30,000 de tráfico de ataque para evitar sesgo en el modelo.
5. Separar características `X` y etiqueta `y`.

In [ ]:
# Cargar el dataset con pandas
df = pd.read_csv('CICIoT2023_xsmall.csv')

# Eliminar la columna de índice sin nombre (primera columna)
df = df.drop(df.columns[0], axis=1)

# Eliminar filas con NaN
df = df.dropna()

# Convertir etiqueta a binaria: 0 = tráfico benigno, 1 = ataque
df['label'] = (df['label'] != 'BenignTraffic').astype(int)

print('Distribución de clases (antes de balancear):')
print(df['label'].value_counts())

# Balancear clases: misma cantidad de ejemplos benignos y de ataque
df_benign = df[df['label'] == 0]
df_attack = df[df['label'] == 1].sample(n=len(df_benign), random_state=42)
df = pd.concat([df_benign, df_attack]).sample(frac=1, random_state=42).reset_index(drop=True)

# Separar características (X) y etiqueta (y)
X = df.drop('label', axis=1).values
y = df['label'].values
m = y.size

print(f'\nNúmero de ejemplos (m): {m}')
print(f'Número de propiedades (n): {X.shape[1]}')
print('\nDistribución final de clases:')
print(df['label'].value_counts())

## 2. Visualización de los Datos

Se grafican dos características del flujo de red para observar la distribución de tráfico benigno y de ataque.
Los puntos `*` representan ataques y los `o` amarillos representan tráfico benigno.

In [ ]:
def plotData(X, y):
    # Grafica los puntos de datos X e y en una nueva figura.
    # Grafica los puntos de datos con * para los positivos (ataque)
    # y o para los negativos (benigno).

    # Crea una nueva figura
    fig = pyplot.figure()

    # Índices de ejemplos positivos y negativos
    pos = y == 1
    neg = y == 0

    # Graficar ejemplos
    pyplot.plot(X[pos, 0], X[pos, 1], 'k*', lw=2, ms=5)
    pyplot.plot(X[neg, 0], X[neg, 1], 'ko', mfc='y', ms=4, mec='k', mew=1)

In [ ]:
# Visualizar la relación entre Rate y Srate para las dos clases
plotData(X[:, [4, 5]], y)
pyplot.xlabel('Rate')
pyplot.ylabel('Srate')
pyplot.legend(['Ataque', 'Benigno'])

## 3. Función Sigmoidea

La hipótesis de la regresión logística es:

$$h_\theta(x) = g(\theta^T x) = \frac{1}{1+e^{-\theta^T x}}$$

La función sigmoidea mapea cualquier valor real al intervalo $(0, 1)$, interpretable como la **probabilidad de que el flujo sea un ataque**.
Valores cercanos a 1 indican alta probabilidad de ataque; cercanos a 0, tráfico benigno.

In [ ]:
def sigmoid(z):
    # Calcula la sigmoide de una entrada z
    # convierte la entrada a un arreglo numpy
    z = np.array(z)

    g = np.zeros(z.shape)

    g = 1 / (1 + np.exp(-z))

    return g

In [ ]:
# Prueba la implementación de la función sigmoid
z = [-100, 0, 0.5, 100]
g = sigmoid(z)
print('g(', z, ') = ', g)

## 4. Normalización y División de Datos

### Normalización: $X_{norm} = \frac{X - \mu}{\sigma}$

Las características del tráfico de red tienen escalas muy diferentes (bytes, segundos, conteos, flags). La normalización es crucial para que el descenso de gradiente converja de forma eficiente.

Luego se agrega el término de intercepción $\theta_0$ (columna de unos) y se divide el dataset en **80% entrenamiento / 20% validación** con permutación aleatoria.

In [ ]:
def featureNormalize(X):

    X_norm = X.copy()
    mu = np.zeros(X.shape[1])
    sigma = np.zeros(X.shape[1])

    mu = np.mean(X, axis=0)
    sigma = np.std(X, axis=0)
    sigma[sigma == 0] = 1  # evitar división por cero
    X_norm = (X - mu) / sigma

    return X_norm, mu, sigma

In [ ]:
# llama featureNormalize con los datos cargados
X_norm, mu, sigma = featureNormalize(X)
print(X_norm[:3])

# Configurar la matriz adecuadamente, y agregar columna de unos (término de intercepción theta_0)
m, n = X.shape
X = np.concatenate([np.ones((m, 1)), X_norm], axis=1)

# División: 80% entrenamiento, 20% validación
np.random.seed(42)
indices = np.random.permutation(m)
m_val = int(m * 0.2)
idx_val = indices[:m_val]
idx_train = indices[m_val:]

X_train = X[idx_train]
y_train = y[idx_train]
X_val = X[idx_val]
y_val = y[idx_val]

print(f'Ejemplos de entrenamiento: {X_train.shape[0]}')
print(f'Ejemplos de validación:    {X_val.shape[0]}')

## 5. Función de Costo $J(\theta)$

Para la regresión logística se utiliza la entropía cruzada binaria (*binary cross-entropy*):

$$J(\theta) = \frac{1}{m} \sum_{i=1}^{m} \left[ -y^{(i)} \log\left(h_\theta\left( x^{(i)} \right) \right) - \left( 1 - y^{(i)}\right) \log \left( 1 - h_\theta\left( x^{(i)} \right) \right) \right]$$

Donde $h_\theta(x) = sigmoid(\theta^T x) \in (0, 1)$ representa la probabilidad estimada de que el flujo sea un ataque.
A diferencia de la regresión lineal, la función de costo aquí es no cuadrática para garantizar convexidad con la hipótesis sigmoidea.

In [ ]:
def calcularCosto(theta, X, y):
    # Inicializar algunos valores utiles
    m = y.size  # numero de ejemplos de entrenamiento

    J = 0
    h = sigmoid(X.dot(theta.T))
    J = (1 / m) * np.sum(-y.dot(np.log(h)) - (1 - y).dot(np.log(1 - h)))

    return J

In [ ]:
# Probar la función de costo con theta inicial en cero
theta = np.zeros(X_train.shape[1])
JJ = calcularCosto(theta, X_train, y_train)
print(f'con theta: {theta[:4]} ... se obtiene un costo de: {JJ}')

## 6. Descenso de Gradiente

El gradiente de la función de costo respecto a $\theta_j$ es:

$$ \frac{\partial J(\theta)}{\partial \theta_j} = \frac{1}{m} \sum_{i=1}^m \left( h_\theta \left( x^{(i)} \right) - y^{(i)} \right) x_j^{(i)} $$

En forma vectorizada: $\theta := \theta - \frac{\alpha}{m} X^T (h - y)$

Donde $\alpha$ es la tasa de aprendizaje. El algoritmo itera actualizando $\theta$ hasta minimizar $J(\theta)$.

In [ ]:
def descensoGradiente(theta, X, y, alpha, num_iters):
    # Inicializa algunos valores
    m = y.shape[0]  # numero de ejemplos de entrenamiento

    # realiza una copia de theta, el cual será actualizada por el descenso por el gradiente
    theta = theta.copy()
    J_history = []

    for i in range(num_iters):
        h = sigmoid(X.dot(theta.T))
        theta = theta - (alpha / m) * (h - y).dot(X)

        J_history.append(calcularCosto(theta, X, y))
    return theta, J_history

In [ ]:
# Elegir algun valor para alpha (probar varias alternativas)
alpha = 0.1
num_iters = 1000

# inicializa theta y ejecuta el descenso por el gradiente
theta = np.zeros(X_train.shape[1])
theta, J_history = descensoGradiente(theta, X_train, y_train, alpha, num_iters)

# Grafica la convergencia del costo
pyplot.plot(np.arange(len(J_history)), J_history, lw=2)
pyplot.xlabel('Numero de iteraciones')
pyplot.ylabel('Costo J')

# Muestra los resultados del descenso por el gradiente
print('theta calculado por el descenso por el gradiente: {:s}'.format(str(theta)))
print(f'\nCosto entrenamiento: {J_history[-1]}')

# Costo en validación
J_val = calcularCosto(theta, X_val, y_val)
print(f'Costo validación:    {J_val}')

## 7. Evaluación y Predicciones

Se evalúa el modelo sobre el conjunto de **validación** (datos que no participaron en el entrenamiento).

Un flujo se clasifica como **ataque (1)** si $h_\theta(x) \geq 0.5$, y como **benigno (0)** en caso contrario.

Se muestra la exactitud global del modelo y una tabla con 120 predicciones individuales comparando el valor real con el predicho.

In [ ]:
# Predicciones sobre el conjunto de validación
prob_val = sigmoid(np.dot(X_val, theta))
pred_val = (prob_val >= 0.5).astype(int)

# Exactitud del modelo
exactitud = np.mean(pred_val == y_val) * 100
print(f'Exactitud en validación: {exactitud:.2f}%')
print(f'Correctamente clasificados: {np.sum(pred_val == y_val)} de {len(y_val)}')

In [ ]:
# 120 predicciones del modelo de regresión logística binaria (>= 100 requeridas)
print('{:>10s}{:>15s}{:>15s}{:>15s}'.format('Ejemplo', 'Real', 'Probabilidad', 'Prediccion'))
print('-' * 55)
for i in range(120):
    prob = sigmoid(np.dot(X_val[i], theta))
    pred = 1 if prob >= 0.5 else 0
    real_label = 'Ataque' if y_val[i] == 1 else 'Benigno'
    pred_label = 'Ataque' if pred == 1 else 'Benigno'
    print('{:>10d}{:>15s}{:>15.4f}{:>15s}'.format(i + 1, real_label, prob, pred_label))